In [2]:
%pip install pandas numpy pyarrow scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import time
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [4]:
Arm = Path("data/cic_ynu/arm_sar.parquet")
Mips = Path("data/cic_ynu/mips_sar.parquet")
Mipsel = Path("data/cic_ynu/mipsel_sar.parquet")
x86 = Path("data/cic_ynu/x86_sar.parquet")

Arm = pd.read_parquet(Arm, engine="pyarrow")
Mips = pd.read_parquet(Mips, engine="pyarrow")
Mipsel = pd.read_parquet(Mipsel, engine="pyarrow")
x86 = pd.read_parquet(x86, engine="pyarrow")

print("ARM:", Arm.shape)
print("MIPS:", Mips.shape)
print("MIPSEL:", Mipsel.shape)
print("x86:", x86.shape)

ARM: (645518, 463)
MIPS: (430540, 394)
MIPSEL: (430540, 394)
x86: (529212, 411)


In [6]:
combined = pd.concat(
    [Arm, Mips, Mipsel, x86],
    axis=0,
    ignore_index=True,
    sort=False
)

print("Combined shape:", combined.shape)
print("\nArchitecture counts:")
print(combined["Arch"].value_counts())

print("\nMalware family counts:")
print(combined["MalwareFamily"].value_counts())

Combined shape: (2035810, 334)

Architecture counts:
Arch
mips    861080
arm     645518
x86     529212
Name: count, dtype: int64

Malware family counts:
MalwareFamily
Benign       1051309
Mirai         781419
Unknown       168603
DarkNexus      23579
Generic         5250
Gafgyt          5171
Tsunami          359
Agent            120
Name: count, dtype: int64


In [7]:
combined = combined.dropna(subset=["MalwareFamily"]).copy()

combined["Label"] = np.where(combined["MalwareFamily"] == "Benign", 0, 1)

print("Label counts:")
print(combined["Label"].value_counts())

print("\n0 = Benign")
print("1 = Malware")

Label counts:
Label
0    1051309
1     984501
Name: count, dtype: int64

0 = Benign
1 = Malware


In [8]:
feature_cols = combined.select_dtypes(include=[np.number]).columns.tolist()

if "Label" in feature_cols:
    feature_cols.remove("Label")

X = combined[feature_cols].replace([np.inf, -np.inf], np.nan).astype("float32")
y = combined["Label"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2035810, 314)
y shape: (2035810,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (1628648, 314)
X_test: (407162, 314)


In [10]:
imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train).astype("float32")
X_test = imputer.transform(X_test).astype("float32")

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

print("Data cleaned and normalized")

Data cleaned and normalized


In [11]:
start = time.time()

model = SGDClassifier(
    loss="hinge",
    max_iter=20,
    tol=1e-3,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)

end = time.time()

print("Training time:", round(end - start, 2), "seconds")

Training time: 27.28 seconds


In [12]:
y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Benign", "Malware"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8749220212102308

Classification Report:
              precision    recall  f1-score   support

      Benign       0.97      0.78      0.87    210262
     Malware       0.80      0.98      0.88    196900

    accuracy                           0.87    407162
   macro avg       0.89      0.88      0.87    407162
weighted avg       0.89      0.87      0.87    407162


Confusion Matrix:
[[163539  46723]
 [  4204 192696]]


In [13]:
out_dir = r"C:\Users\Ssanp\OneDrive\Documents\SRP 2026"

big_classical_results = pd.DataFrame([{
    "Dataset": "Big SAR",
    "Model": "Classical SVM",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "F1 Score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
}])

big_classical_results.to_csv(out_dir + r"\big_classical_results.csv", index=False)

big_classical_predictions = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

big_classical_predictions.to_csv(out_dir + r"\big_classical_predictions.csv", index=False)

big_classical_results

,Dataset,Model,Accuracy,Precision,Recall,F1 Score
0,Big SAR,Classical SVM,0.874922,0.892684,0.874922,0.873982
